# Lab: Large Context Window vs RAG (GitHub Models)

In this lab you will learn:

1. What a context window is
2. How to place documents directly inside the prompt
3. How to count tokens in a prompt
4. How large context models sometimes remove the need for RAG

This notebook is designed for students using GitHub Models or OpenAI compatible APIs.

# Step 1 — Install Libraries

In [1]:
!pip install openai tiktoken

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/879.1 kB ? eta -:--:--
   ---------------------------------------- 879.1/879.1 kB 6.6 MB/s  0:00:00

   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   -------- ------------------------------- 1/5 [regex]
   ---------------- ----------------------- 2/5 [charset_normalizer]
   ---------------- ----------------------- 2/5 [charset_normalizer]
   ------------------------ --------------- 3/5 [requests]
   ------------------------ --------------- 3/5 [requests]
   -------------------------------- ------- 4/5 [tiktoken]
   ---------------------------------------- 5/5 [tiktoken]




[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\hetarra\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


# Step 2 — Configure GitHub Models / OpenAI Client

In [ ]:
import os
from openai import OpenAI

client = OpenAI(
    base_url='https://models.inference.ai.azure.com',
    api_key=os.getenv('GITHUB_TOKEN')
)

In [2]:
# GitHub Models Configuration
import json
import os
from openai import OpenAI
from dotenv import load_dotenv

# Load environment variables
load_dotenv(dotenv_path="../.env")

# Settings
github_token = os.getenv("GITHUB_TOKEN")
deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")

# Initialize client
client = OpenAI(
    base_url="https://models.github.ai/inference",
    api_key=github_token
)

print("GitHub Models client configured successfully!")

GitHub Models client configured successfully!


# Step 3 — Example Document (Context Data)

Instead of retrieving documents from a vector database (RAG), we will place the document directly inside the prompt.

In [3]:
context_document = '''
Company Vacation Policy

Employees receive 20 days of paid vacation annually.
Unused vacation days can be carried forward for one year.
Vacation requests must be submitted at least two weeks in advance.
Managers must approve all leave requests.
'''

# Step 4 — User Question

In [4]:
question = 'How many vacation days do employees receive each year?'

# Step 5 — Build Prompt With Context

In [5]:
prompt = f'''
You are an HR assistant.

CONTEXT:
{context_document}

QUESTION:
{question}
'''

print(prompt)


You are an HR assistant.

CONTEXT:

Company Vacation Policy

Employees receive 20 days of paid vacation annually.
Unused vacation days can be carried forward for one year.
Vacation requests must be submitted at least two weeks in advance.
Managers must approve all leave requests.


QUESTION:
How many vacation days do employees receive each year?



# Step 6 — Count Tokens in Prompt

In [6]:
import tiktoken

enc = tiktoken.encoding_for_model('gpt-4o')
tokens = enc.encode(prompt)

print('Prompt token count:', len(tokens))

Prompt token count: 66


# Step 7 — Ask the Model

In [7]:
response = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': prompt}]
)

print(response.choices[0].message.content)

Employees receive 20 days of paid vacation annually.


# Step 8 — Understanding Context Window

Example context windows:

| Model | Context Window |
|------|------|
| GPT-4o | 128K tokens |
| GPT-4o mini | 128K tokens |
| Claude 3 | ~200K tokens |
| Gemini 1.5 | up to 1M tokens |

Rule:

input tokens + output tokens <= context window